In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import shapiro, probplot
import matplotlib.pyplot as plt

# ==============================================================================
# ПАРАМЕТРИ ВАРІАНТА 6 (м. Чернігів)
# ==============================================================================
VARIANT = 6
CITY = "Чернігів"
MEAN_TEMP = 8.0
AMPLITUDE = 13.0
HUM_BASE = 75.0

np.random.seed(42)

# Генерація вихідного набору даних
rows = []
for month in range(1, 13):
    temp = MEAN_TEMP + AMPLITUDE * np.cos((month - 7) / 12 * 2 * np.pi) + np.random.normal(0, 1.0)
    humidity = HUM_BASE + np.random.normal(0, 5.0)

    if temp < 8:
        opal_days = 30       # опалювальний період триває весь місяць
    elif temp < 12:
        opal_days = 15       # опалювальний період триває частину місяця
    else:
        opal_days = 0        # опалення не потрібне

    cost = (
        200
        - 9.0 * temp
        + 0.6 * humidity
        + 2.5 * opal_days
        + np.random.normal(0, 15)
    )
    rows.append({
        "місто": CITY,
        "місяць": month,
        "температура": round(temp, 1),
        "вологість": round(humidity, 1),
        "опалювальні_дні": opal_days,
        "витрати_на_опалення": round(cost, 1),
    })

heating = pd.DataFrame(rows)

print(f"=== ПРАКТИКА 18: Варіант {VARIANT} ({CITY}) ===")
print(heating.to_string(index=False))

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 1. Модель із двома предикторами (температура + вологість)
# ------------------------------------------------------------------------------
print("\n=== Завдання 1: Модель 1 (температура + вологість) ===")

X1 = heating[["температура", "вологість"]]
X1_const = sm.add_constant(X1)
y = heating["витрати_на_опалення"]

model1 = sm.OLS(y, X1_const).fit()
print(model1.summary())

r2_m1 = model1.rsquared
adj_r2_m1 = model1.rsquared_adj

print(f"\nМодель 1: R² = {r2_m1:.4f}, Adjusted R² = {adj_r2_m1:.4f}")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. Інтерпретація коефіцієнтів у реальних одиницях
# ------------------------------------------------------------------------------
print("\n=== Завдання 2: Коефіцієнти Моделі 1 ===")
for var in ["температура", "вологість"]:
    coef = model1.params[var]
    pval = model1.pvalues[var]
    sig = "статистично значущий (p < 0.05)" if pval < 0.05 else "НЕ є статистично значущим (p >= 0.05)"
    print(f"Предиктор '{var}': coef = {coef:.4f}, p-value = {pval:.4e} -> {sig}")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 3. Перевірка мультиколінеарності через VIF (Модель 1)
# ------------------------------------------------------------------------------
print("\n=== Завдання 3: VIF для Моделі 1 ===")
for i, col in enumerate(X1_const.columns):
    if col == "const":
        continue
    vif_val = variance_inflation_factor(X1_const.values, i)
    print(f"VIF('{col}') = {vif_val:.2f}")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 4. Третій предиктор: R² проти adjusted R² (Модель 2)
# ------------------------------------------------------------------------------
print("\n=== Завдання 4: Модель 2 (температура + вологість + опалювальні_дні) ===")

X2 = heating[["температура", "вологість", "опалювальні_дні"]]
X2_const = sm.add_constant(X2)

model2 = sm.OLS(y, X2_const).fit()
print(model2.summary())

r2_m2 = model2.rsquared
adj_r2_m2 = model2.rsquared_adj

print(f"\nПорівняння метрик:")
print(f"Модель 1 (2 предиктори): R² = {r2_m1:.4f}, Adj R² = {adj_r2_m1:.4f}")
print(f"Модель 2 (3 предиктори): R² = {r2_m2:.4f}, Adj R² = {adj_r2_m2:.4f}")

print("\nVIF для Моделі 2 (3 предиктори):")
for i, col in enumerate(X2_const.columns):
    if col == "const":
        continue
    vif_val = variance_inflation_factor(X2_const.values, i)
    print(f"VIF('{col}') = {vif_val:.2f}")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 5. Діагностика залишків (Модель 1)
# ------------------------------------------------------------------------------
print("\n=== Завдання 5: Діагностика залишків (Модель 1) ===")

residuals1 = model1.resid
fitted1 = model1.fittedvalues

# Графіки: Залишки vs Передбачені значення та QQ-plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Residuals vs Fitted
ax1.scatter(fitted1, residuals1, color="blue", s=50)
ax1.axhline(0, color="red", linestyle="--")
ax1.set_xlabel("Передбачені значення (Fitted values)")
ax1.set_ylabel("Залишки (Residuals)")
ax1.set_title("Графік залишків vs Передбачені")
ax1.grid(True, linestyle="--", alpha=0.6)

# QQ-plot
probplot(residuals1, dist="norm", plot=ax2)
ax2.set_title("Q-Q plot залишків")
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("residuals_diagnostics_v6.png")
plt.close()

# Тест Шапіро-Вілка
shapiro_stat, shapiro_p = shapiro(residuals1)
print(f"Тест Шапіро-Вілка на нормальність залишків: W = {shapiro_stat:.4f}, p-value = {shapiro_p:.4f}")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 6. Порівняння з простою регресією (Практика 17)
# ------------------------------------------------------------------------------
print("\n=== Завдання 6: Порівняння з простою регресією (лише температура) ===")

X_simple = sm.add_constant(heating[["температура"]])
model_simple = sm.OLS(y, X_simple).fit()

r2_simple = model_simple.rsquared

print(f"Проста регресія (температура):   R² = {r2_simple:.4f}")
print(f"Множинна регресія (темп+волог):  Adj R² = {adj_r2_m1:.4f} (R² = {r2_m1:.4f})")

=== ПРАКТИКА 18: Варіант 6 (Чернігів) ===
   місто  місяць  температура  вологість  опалювальні_дні  витрати_на_опалення
Чернігів       1         -4.5       74.3               30                369.8
Чернігів       2         -1.7       73.8               30                331.4
Чернігів       3          3.1       78.8               30                287.5
Чернігів       4          8.5       72.7               15                197.2
Чернігів       5         14.7       65.4                0                 80.7
Чернігів       6         18.7       69.9                0                 78.4
Чернігів       7         20.1       67.9                0                 81.9
Чернігів       8         19.0       75.3                0                 52.5
Чернігів       9         14.0       75.6                0                102.5
Чернігів      10          8.4       72.0               15                200.9
Чернігів      11          0.9       84.3               30                317.3
Чернігів  

# Відповіді на завдання та контрольні питання
**Практична робота №18: Множинна лінійна регресія: відбір предикторів, інтерпретація коефіцієнтів, діагностика залишків**
**Варіант 6:** м. Чернігів

---

## Виконання завдань

### Завдання 1. Модель із двома предикторами
Двопредикторна модель побудована для залежної змінної `витрати_на_опалення` за предикторами `температура` та `вологість`.
* **Рівняння регресії:**
  $$\text{витрати} = 227.8188 - 11.2386 \times \text{температура} + 0.3831 \times \text{вологість}$$
* **Метрики якості:**
  * $R^2 = 0.9413$ (94.13% варіації витрат пояснюється моделлю)
  * $\text{Adjusted } R^2 = 0.9282$

---

### Завдання 2. Інтерпретація коефіцієнтів у реальних одиницях
1. **Температура ($b_1 = -11.2386$, $P>\vert{}t\vert{} < 0.0001$):**
   За незмінної вологості повітря, підвищення середньомісячної температури на $1^\circ\text{C}$ зменшує очікувані витрати на опалення в середньому на **11.24 умовних одиниць**. Оскільки $p < 0.05$, цей предиктор є **статистично значущим** при $\alpha = 0.05$.
2. **Вологість ($b_2 = 0.3831$, $P>\vert{}t\vert{} = 0.6279$):**
   За незмінної температури, підвищення відносної вологості на $1\%$ збільшує очікувані витрати на опалення в середньому на **0.38 умовних одиниць**. Оскільки $p = 0.628 \ge 0.05$, цей предиктор **не є статистично значущим** за інших рівних умов.
3. **Константа ($b_0 = 227.8188$, $P>\vert{}t\vert{} = 0.0101$):**
   Базові витрати на опалення за умовних $0^\circ\text{C}$ температури та $0\%$ вологості.

---

### Завдання 3. Перевірка мультиколінеарності через VIF
* $\text{VIF}(\text{температура}) = 1.00$
* $\text{VIF}(\text{вологість}) = 1.00$

**Оцінка:** Обидва значення VIF значно менші за практичний поріг 5 (і тим більше 10). Проблеми мультиколінеарності між температурою та вологістю немає.
**Пояснення з коду:** У формулі генерації даних вологість задавалася як незалежний нормальний шум навколо базового значення (`hum_base + normal(0, 5.0)`) і ніяк не функціонально не пов'язувалася з температурою.

---

### Завдання 4. Третій предиктор: R² проти adjusted R²

| Модель | Предиктори | $R^2$ | Adjusted $R^2$ |
| :--- | :--- | :---: | :---: |
| **Модель 1** | температура, вологість | $0.9413$ | $0.9282$ |
| **Модель 2** | температура, вологість, опалювальні_дні | $0.9427$ | $0.9213$ |

* **Аналіз:** Звичайний $R^2$ зростав несуттєво (з $0.9413$ до $0.9427$), тоді як скоригований $\text{Adjusted } R^2$ **зменшився** з $0.9282$ до $0.9213$.
* **VIF у Моделі 2:** $\text{VIF}(\text{температура}) = 11.52$, $\text{VIF}(\text{опалювальні\_дні}) = 11.52$, $\text{VIF}(\text{вологість}) = 1.00$.
* **Висновок:** Додавання предиктора `опалювальні_дні` **не є виправданим**. Цей предиктор перебуває у сильній мультиколінеарності з температурою ($\text{VIF} > 10$), оскільки дискретний показник днів прямо обчислювався з температурних порогів ($<8^\circ\text{C}$, $<12^\circ\text{C}$). Новий предиктор не додає унікальної інформації, а штраф adjusted $R^2$ за втрату ступеня свободи перевищує мінімальний приріст поясненої дисперсії.

---

### Завдання 5. Діагностика залишків (Модель 1)
1. **Графік залишків vs Передбачені:** Залишки розкидані випадковим чином навколо нульової лінії без вираженої «лійки» чи симетричного розширення/звуження. Ознак гетероскедастичності не виявлено (припущення гомоскедастичності виконується).
2. **Q-Q plot та Тест Шапіро-Вілка:** Точки на Q-Q plot лежать близько до теоретичної прямої нормального розподілу.
   * Статистика Шапіро-Вілка: $W = 0.9492$
   * $p\text{-value} = 0.6128$
   * Оскільки $p = 0.6128 > 0.05$, **немає підстав відхилити нульову гіпотезу** про нормальний розподіл залишків.

---

### Завдання 6. Порівняння з простою регресією
* **Проста регресія (лише температура):** $R^2 = 0.9398$
* **Двопредикторна регресія (температура + вологість):** $\text{Adjusted } R^2 = 0.9282$ ($R^2 = 0.9413$)

**Пояснення:** Додавання вологості збільшило звичайний $R^2$ лише на $0.0015$ (з $93.98\%$ до $94.13\%$), але оскільки вологість не є статистично значущим предиктором ($p = 0.628$), додавання цієї змінної призвело до **зниження скоригованого $\text{Adjusted } R^2$** відносно простої моделі. З погляду МНК, температура сама по собі повністю пояснює майже весь лінійний тренд у витратах, а додаткова змінна виступає лише як шумова складова.

---

## Відповіді на контрольні питання

### 1. Чому `sm.add_constant()` є обов'язковим кроком перед `sm.OLS()`, і що станеться з моделлю, якщо його пропустити?
За замовчуванням у бібліотеці `statsmodels` модель `OLS` **не включає** вільний член (перетин $b_0$, константу).
* Якщо пропустити `sm.add_constant()`, модель буде примусово проходити через початок координат $(0,0)$ (тобто $b_0 = 0$).
* Це призведе до викривлення коефіцієнтів нахилу, зміщення оцінок МНК та неадекватного обчислення $R^2$ (який у `statsmodels` для регресії без константи обчислюється за відмінною формулою і може давати некоректно завищені значення).

---

### 2. У чому коефіцієнт множинної регресії відрізняється від коефіцієнта простої регресії з тим самим предиктором?
* Коефіцієнт **простої регресії** вимірює загальну (парну) асоціацію між $X_1$ та $Y$, включаючи непрямий вплив інших змінних, які корелюють з $X_1$.
* Коефіцієнт **множинної регресії** $b_1$ вимірює **чистий (умовно-ізольований) вплив** $X_1$ на $Y$ **«за інших рівних умов»** (holding other variables constant), усуваючи лінійний ефект усіх інших включених у модель предикторів.

---

### 3. Чому звичайний $R^2$ не може зменшитись при додаванні будь-якого нового предиктора, і чому це робить його непридатним для порівняння моделей з різною кількістю предикторів?
Метод найменших квадратів (МНК) мінімізує суму квадратів залишків $SS_{\text{res}}$. Додавання нового предиктора надає алгоритму МНК додатковий ступінь вільності під час оптимізації, тому $SS_{\text{res}}$ може лише зменшитися або залишитися незмінною ($SS_{\text{res, new}} \le SS_{\text{res, old}}$).
Відповідно, $R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}}$ гарантовано **не зменшується**. Через це звичайний $R^2$ штучно віддає перевагу більш складним моделям з багатьма неефективними регресорами. Для порівняння моделей різної розмірності використовують **Adjusted $R^2$**, який вводить штраф за кожну додатково введену змінну:
$$\text{Adj } R^2 = 1 - (1 - R^2) \frac{n - 1}{n - k - 1}$$

---

### 4. Що означає VIF предиктора $\approx 1$, і що означає VIF $> 10$ — і що з этим варто зробити на практиці?
* **$\text{VIF} \approx 1$:** Повна відсутність мультиколінеарності. Даний предиктор ортогональний (нелінійно зв'язаний) до інших регресорів моделі. Оцінки коефіцієнтів є стабільними.
* **$\text{VIF} > 10$:** Критична мультиколінеарність. Предиктор майже повністю дублює інформацію інших змінних. Це призводить до роздування стандартних помилок коефіцієнтів ($SE(b_j)$ зростає в $\sqrt{\text{VIF}}$ разів), роблячи оцінки коефіцієнтів нестабільними, а їхні $p$-value — ненадійними.
* **Що робити на практиці:**
  1. Вилучити один із дублюючих предикторів з моделі (зазвичай той, що менше обґрунтований змістовно або має гірший зв'язок з $Y$).
  2. Об'єднати скорельовані змінні в єдиний індекс/композитний показник.
  3. Використати методи зниження розмірності (наприклад, метод головних компонент — PCA) або регуляризовану регресію (Ridge / Lasso).